# ENIGMA ML Competition: horas de interrupción por zona

Solución a la competencia ENIGMA (Kaggle). Hay que predecir las horas de interrupción del servicio eléctrico por usuario (`target`) de cada zona durante los 12 meses siguientes al final del historial (`t = 233..244`). La métrica es RMSLE, que es lo mismo que el RMSE sobre `log1p(target)`.

**Cómo está armada**

El test mezcla dos situaciones distintas y por eso se modelan por separado:

- **Zonas con historia** (188 zonas, 82 % de las filas): LightGBM entrenado sobre un dataset de varios *orígenes* en el tiempo. En cada origen las features usan solo datos anteriores a él, que es exactamente lo que ocurre en el test (origen `t = 232`).
- **Zonas sin historia** (42 zonas, 18 % de las filas): no existen en `train`, así que se parte del nivel de su empresa y se corrige con un modelo de residuo basado en la infraestructura de la zona.

Cada bloque cierra con una calibración aditiva pequeña del nivel, en escala log.

**Nota sobre la submission de la competencia.** La que se envió (puesto 1, RMSLE 0.82468 con el 100 % del test) mezclaba además un 27 % de un ensamble anterior para las zonas con historia. Ese ensamble venía de una cadena larga de experimentos que no se puede reproducir de forma limpia, así que este notebook usa solo el modelo final. Según la validación, la mezcla aportaba menos de 0.001.

**Contenido**

1. Configuración y datos
2. Exploración rápida
3. Features estáticas por zona
4. Features históricas sin fuga
5. Zonas con historia: validación
6. Zonas con historia: modelo final
7. Zonas sin historia
8. Armado de la submission
9. Lo que se probó y no funcionó

Para correrlo solo hacen falta los CSV de la competencia en la carpeta `data/`. Completo tarda un par de minutos en una CPU normal.

## 1. Configuración y datos

El notebook busca los CSV en `data/` (o en `../data/`). Si se corre en Kaggle usa la ruta de la competencia.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import QuantileTransformer

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 140)

CANDIDATAS = [Path('data'), Path('../data'),
              Path('/kaggle/input/competitions/enigma-ml-competition/datos-publicos')]
DATA_DIR = next((p for p in CANDIDATAS if (p / 'train.csv').exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError('No encuentro train.csv: copia los CSV de la competencia en data/')

OUT_DIR = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)
print('datos en:', DATA_DIR)

datos en: data


In [2]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
substations = pd.read_csv(DATA_DIR / 'substations.csv')
consumption = pd.read_csv(DATA_DIR / 'consumption.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

for nombre, df in [('train', train), ('test', test), ('substations', substations),
                   ('consumption', consumption), ('sample_submission', sample_sub)]:
    print(f'{nombre:18s} {df.shape}')

train              (41569, 9)
test               (2715, 5)
substations        (135154, 8)
consumption        (125994, 6)
sample_submission  (2715, 2)


## 2. Exploración rápida

Lo importante de esta etapa son dos cosas: cuántas zonas del test no tienen historia, y cómo se distribuye el target (tiene cola larga y muchos ceros).

In [3]:
zonas_train, zonas_test = set(train.zone_id), set(test.zone_id)
zonas_frias = zonas_test - zonas_train
filas_frias = test.zone_id.isin(zonas_frias)

print(f'zonas en test: {len(zonas_test)} | con historia: {len(zonas_test) - len(zonas_frias)} | frías: {len(zonas_frias)}')
print(f'filas frías: {filas_frias.sum()} ({filas_frias.mean():.1%} del test)')
print('t en train:', train.t.min(), '-', train.t.max(), '| t en test:', test.t.min(), '-', test.t.max())
print('inicio (t) de las zonas frías en test:', sorted(test[filas_frias].groupby('zone_id').t.min().unique()))
print(f'empresas de test que no están en train: {len(set(test.company_id) - set(train.company_id))}')

print('\nDistribución del target:')
print(train.target.describe(percentiles=[.5, .9, .99, .995]).round(2).to_string())
print(f'\nmeses sin interrupciones (target = 0): {(train.target == 0).mean():.1%}')

zonas en test: 230 | con historia: 188 | frías: 42


filas frías: 500 (18.4% del test)
t en train: 1 - 232 | t en test: 233 - 244
inicio (t) de las zonas frías en test: [np.int64(233)]
empresas de test que no están en train: 1

Distribución del target:
count    41569.00
mean         5.59
std         14.09
min          0.00
50%          2.26
90%         12.78
99%         45.20
99.5%       66.41
max        685.90

meses sin interrupciones (target = 0): 11.7%


El percentil 99.5 del target está cerca de las 60 horas y el máximo llega a 685. Para que unos pocos meses extremos no dominen el entrenamiento, las etiquetas se recortan a 60 horas. La evaluación, en cambio, siempre se hace contra el target original sin recortar.

## 3. Features estáticas por zona

Resumen por zona de `substations.csv` (capacidad, demanda, tensión, ubicación) y de `consumption.csv` (clientes, consumo, composición por segmento). Son rasgos que casi no cambian en el tiempo.

Se dejan fuera las variables calculadas "a `t = 232`" (crecimiento de clientes, último mes): para un origen antiguo mirarían al futuro. Las zonas sin datos de infraestructura o consumo quedan con la mediana de cada columna.

In [4]:
def tabla_estatica(substations, consumption):
    """Una fila por zona con rasgos de infraestructura y de consumo."""
    sub = pd.concat([substations, pd.get_dummies(substations['voltage_class'], prefix='volt')], axis=1)
    volt_cols = [c for c in sub.columns if c.startswith('volt_')]

    infra = sub.groupby('zone_id').agg(
        sub_count=('capacity', 'count'), cap_sum=('capacity', 'sum'), cap_mean=('capacity', 'mean'),
        cap_std=('capacity', 'std'), cap_max=('capacity', 'max'),
        demand_sp_sum=('demand_sp', 'sum'), demand_ap_sum=('demand_ap', 'sum'),
        ownership_A_ratio=('ownership', lambda s: (s == 'A').mean()),
        x_mean=('x', 'mean'), y_mean=('y', 'mean'),
        x_min=('x', 'min'), x_max=('x', 'max'), y_min=('y', 'min'), y_max=('y', 'max'),
        x_std=('x', 'std'), y_std=('y', 'std'),
        **{c: (c, 'mean') for c in volt_cols}).reset_index()
    infra['demand_total'] = infra['demand_sp_sum'] + infra['demand_ap_sum']
    infra['stress_ratio'] = infra['demand_total'] / (infra['cap_sum'] + 1e-5)
    infra['x_span'] = infra['x_max'] - infra['x_min']
    infra['y_span'] = infra['y_max'] - infra['y_min']
    infra['geo_area'] = infra['x_span'] * infra['y_span']
    infra['geo_density'] = infra['sub_count'] / (infra['geo_area'] + 1.0)
    infra['geo_dispersion'] = np.sqrt(infra['x_std'].fillna(0) ** 2 + infra['y_std'].fillna(0) ** 2)

    cons = consumption.assign(total_kwh=consumption['customers'] * consumption['kwh_per_customer'])
    por_mes = cons.groupby(['zone_id', 't']).agg(total_customers=('customers', 'sum'),
                                                 total_kwh=('total_kwh', 'sum')).reset_index()
    por_mes['kwh_per_cust'] = por_mes['total_kwh'] / (por_mes['total_customers'] + 1e-5)
    consumo = por_mes.groupby('zone_id').agg(
        cust_mean=('total_customers', 'mean'), cust_std=('total_customers', 'std'),
        cust_max=('total_customers', 'max'), kwh_mean=('total_kwh', 'mean'),
        kwh_cust_mean=('kwh_per_cust', 'mean')).reset_index()

    # reparto de clientes por segmento
    seg = pd.get_dummies(cons['segment'], prefix='seg').astype(float)
    seg = seg.mul(cons['customers'], axis=0)
    seg['zone_id'] = cons['zone_id']
    seg = seg.groupby('zone_id').sum()
    seg = seg.div(seg.sum(axis=1) + 1e-5, axis=0).add_suffix('_ratio').reset_index()

    estatica = infra.merge(consumo, on='zone_id', how='outer').merge(seg, on='zone_id', how='left')
    for c in estatica.columns.drop('zone_id'):
        estatica[c] = estatica[c].fillna(estatica[c].median())
    return estatica


static_df = tabla_estatica(substations, consumption)
STATIC_COLS = [c for c in static_df.columns if c != 'zone_id']
print(f'{static_df.shape[0]} zonas | {len(STATIC_COLS)} variables estáticas')

842 zonas | 38 variables estáticas


## 4. Features históricas sin fuga

En el test todas las features se calculan "como de `t = 232`" y se predicen los 12 meses siguientes. Para que el modelo vea esa misma situación al entrenar, se arma un dataset con **varios orígenes** (cada 6 meses, de `t = 100` a `t = 220`). Para cada origen `o`:

- las features se calculan solo con filas de `train` con `t <= o`;
- el objetivo son las filas de `t` entre `o + 1` y `o + 12`.

Entre las features hay medias por zona (histórica y de los últimos 12, 24 y 60 meses), la media zona × mes suavizada hacia la media de la zona, rezagos de 12 y 24 meses, medias de la empresa y la edad de la zona. Si la zona no existe en el origen, esas columnas quedan vacías y `is_cold` vale 1.

In [5]:
def asof_features(hist, objetivo, origen):
    """Features de cada fila de `objetivo` calculadas solo con `hist` hasta `t <= origen`."""
    h = hist[hist.t <= origen].copy()
    h['lt'] = np.log1p(h.target.clip(upper=60))
    h['lf'] = np.log1p(h.freq)
    for c in 'abc':
        h['l' + c] = np.log1p(h['target_' + c].clip(upper=60))
    h['zero'] = (h.target == 0).astype(float)

    # historia completa de cada zona
    zs = h.groupby('zone_id').agg(
        z_mean=('lt', 'mean'), z_med=('lt', 'median'), z_std=('lt', 'std'), z_n=('lt', 'size'),
        z_zero=('zero', 'mean'), z_f=('lf', 'mean'), z_a=('la', 'mean'), z_b=('lb', 'mean'),
        z_c=('lc', 'mean'), z_first=('t', 'min')).reset_index()

    # ventanas recientes
    for k, nombre in [(12, 'r12'), (24, 'r24'), (60, 'r60')]:
        r = h[h.t > origen - k].groupby('zone_id').agg(**{
            nombre + '_mean': ('lt', 'mean'), nombre + '_f': ('lf', 'mean'),
            nombre + '_a': ('la', 'mean'), nombre + '_b': ('lb', 'mean'),
            nombre + '_c': ('lc', 'mean')}).reset_index()
        zs = zs.merge(r, on='zone_id', how='left')

    # zona x mes, encogida hacia la media de la zona (peso 5)
    zm = h.groupby(['zone_id', 'month'])['lt'].agg(zm_mean='mean', zm_n='size').reset_index()
    zm = zm.merge(zs[['zone_id', 'z_mean']], on='zone_id')
    zm['zm_smooth'] = (zm.zm_n * zm.zm_mean + 5 * zm.z_mean) / (zm.zm_n + 5)
    zm = zm[['zone_id', 'month', 'zm_smooth', 'zm_n']]

    # empresa: respaldo para zonas sin historia
    cm = h[h.t > origen - 60].groupby(['company_id', 'month'])['lt'].median().rename('cm_med').reset_index()
    c24 = h[h.t > origen - 24].groupby('company_id')['lt'].mean().rename('c_mean24').reset_index()

    out = objetivo.copy()
    out['h'] = out.t - origen
    out = (out.merge(zs, on='zone_id', how='left')
              .merge(zm, on=['zone_id', 'month'], how='left')
              .merge(cm, on=['company_id', 'month'], how='left')
              .merge(c24, on='company_id', how='left'))

    # rezagos de 12 y 24 meses respecto a la fila objetivo (solo si caen antes del origen)
    por_zona_t = h.set_index(['zone_id', 't'])['lt']
    for k in (12, 24):
        idx = pd.MultiIndex.from_arrays([out.zone_id, out.t - k])
        v = por_zona_t.reindex(idx).values
        out[f'lag{k}'] = np.where(out.t.values - k <= origen, v, np.nan)

    out['age'] = origen - out.z_first
    out['is_cold'] = out.z_n.isna().astype(int)
    out['sin_m'] = np.sin(2 * np.pi * out.month / 12)
    out['cos_m'] = np.cos(2 * np.pi * out.month / 12)
    return out


def construir_origenes(train, origenes):
    partes = []
    for o in origenes:
        objetivo = train[(train.t > o) & (train.t <= o + 12)]
        f = asof_features(train, objetivo, o)
        f['origin'] = o
        partes.append(f)
    return pd.concat(partes, ignore_index=True)


ORIGENES = list(range(100, 221, 6))
ds = construir_origenes(train, ORIGENES)
print(ds.shape, '| filas de zonas sin historia en el origen:', f'{ds.is_cold.mean():.1%}')

(47903, 46) | filas de zonas sin historia en el origen: 1.6%


**Control anti-fuga.** Si multiplico por 7 todo lo que ocurre después del origen, las features de ese origen no pueden cambiar. Si cambian, se está colando información del futuro.

In [6]:
def hay_fuga(train, origen=208):
    objetivo = train[(train.t > origen) & (train.t <= origen + 12)]
    base = asof_features(train, objetivo, origen)

    alterado = train.copy()
    cols = ['target', 'target_a', 'target_b', 'target_c', 'freq']
    alterado.loc[alterado.t > origen, cols] *= 7
    nuevo = asof_features(alterado, objetivo, origen)

    numericas = [c for c in base.select_dtypes('number').columns if c not in cols]
    return not np.allclose(base[numericas].fillna(-1), nuevo[numericas].fillna(-1))


assert not hay_fuga(train), 'las features usan información posterior al origen'
print('sin fuga de información futura')

sin fuga de información futura


## 5. Zonas con historia: validación

**Protocolo.** Para validar en el origen `vo` se entrena solo con orígenes cuyo horizonte ya terminó (`origen + 12 <= vo`) y se predice el bloque `vo + 1 .. vo + 12`. Así se simula lo que pasará en el test. Se repite en siete orígenes (148, 160, ..., 220), que corresponden a siete años consecutivos y sin solaparse. Un solo origen es demasiado ruidoso: entre años el error cambia en ±0.04, más que casi cualquier mejora que se quiera medir.

**Modelo.** LightGBM sobre `log1p(target)` recortado a 60 horas, con peso decreciente para los orígenes más antiguos. Dos cosas resultaron decisivas en los experimentos:

- el error **sube** con el número de rondas (con 300 rondas ya hay sobreajuste), porque la señal es débil frente al ruido mensual;
- `extra_trees` junto con una regularización fuerte (`lambda_l2`, `min_child_samples`, `colsample_bytree`) mejora en los siete orígenes.

La referencia de comparación es la media zona × mes suavizada (`zm_smooth`), que es la predicción "ingenua" más fuerte.

In [7]:
DYN = ['h', 'age', 'is_cold', 'sin_m', 'cos_m',
       'z_mean', 'z_med', 'z_std', 'z_n', 'z_zero', 'z_f', 'z_a', 'z_b', 'z_c',
       'r12_mean', 'r12_f', 'r12_a', 'r12_b', 'r12_c',
       'r24_mean', 'r24_f', 'r24_a', 'r24_b', 'r24_c',
       'r60_mean', 'r60_f', 'r60_a', 'r60_b', 'r60_c',
       'zm_smooth', 'zm_n', 'cm_med', 'c_mean24', 'lag12', 'lag24']
FEATS = DYN + STATIC_COLS

# lista explícita de columnas: nada de target, origin ni t entra como feature
assert not {'target', 'target_a', 'target_b', 'target_c', 'freq', 'origin', 't'} & set(FEATS)

TOPE_HORAS = 60.0
ds = ds.merge(static_df, on='zone_id', how='left')
ds['y_tot'] = np.log1p(ds['target'].clip(0, TOPE_HORAS))
print('features:', len(FEATS), '| estáticas:', len(STATIC_COLS))

features: 73 | estáticas: 38


In [8]:
PARAMS = dict(objective='regression', metric='rmse', learning_rate=0.03, num_leaves=25, max_depth=6,
              min_child_samples=200, subsample=0.8, subsample_freq=1, colsample_bytree=0.5,
              lambda_l2=100.0, extra_trees=True, verbose=-1)
N_RONDAS = 200
DECAY = 0.007      # peso exp(-DECAY * antigüedad_en_orígenes) para el entrenamiento


def entrenar_predecir(entrena, predice, semillas, n_rondas=N_RONDAS):
    """Entrena con `entrena` (promedio de semillas) y devuelve la predicción en log1p para `predice`."""
    peso = np.exp(-DECAY * (entrena['origin'].max() - entrena['origin'].values))
    X, Xp = entrena[FEATS].astype(np.float32), predice[FEATS].astype(np.float32)
    pred = np.zeros(len(predice))
    for s in semillas:
        prm = dict(PARAMS, seed=s, bagging_seed=s, feature_fraction_seed=s)
        modelo = lgb.train(prm, lgb.Dataset(X, entrena['y_tot'].values, weight=peso), n_rondas)
        pred += modelo.predict(Xp) / len(semillas)
    return pred


def en_log(pred_log):
    """Predicción en log1p con piso en cero (el target no puede ser negativo)."""
    return np.log1p(np.clip(np.expm1(pred_log), 0, None))

In [9]:
ORIGENES_VAL = (148, 160, 172, 184, 196, 208, 220)
filas = []
for vo in ORIGENES_VAL:
    entrena = ds[ds.origin + 12 <= vo]
    valida = ds[(ds.origin == vo) & (ds.is_cold == 0)]       # solo zonas con historia
    pred = en_log(entrenar_predecir(entrena, valida, semillas=(42, 123)))
    y = np.log1p(valida['target'].values)
    ingenua = valida['zm_smooth'].fillna(valida['cm_med']).fillna(1.2).values

    filas.append({'origen': vo, 'filas': len(valida),
                  'modelo': np.sqrt(np.mean((pred - y) ** 2)),
                  'zona-mes': np.sqrt(np.mean((ingenua - y) ** 2)),
                  'sesgo': np.mean(pred - y)})
    print(f'origen {vo}: {filas[-1]["modelo"]:.4f}', flush=True)

validacion = pd.DataFrame(filas).set_index('origen')
print()
print(validacion.round(4).to_string())
print(f"\nRMSLE medio, 7 orígenes: modelo {validacion['modelo'].mean():.4f} | zona-mes {validacion['zona-mes'].mean():.4f}")
print(f"RMSLE medio, últimos 3 (196/208/220): modelo {validacion.loc[[196, 208, 220], 'modelo'].mean():.4f}")

origen 148: 0.8790


origen 160: 0.7990


origen 172: 0.8321


origen 184: 0.8132


origen 196: 0.8269


origen 208: 0.7950


origen 220: 0.8220



        filas  modelo  zona-mes   sesgo
origen                                 
148      2316  0.8790    0.9209  0.0807
160      2317  0.7990    0.8641  0.0581
172      2321  0.8321    0.8915  0.0251
184      2168  0.8132    0.9273  0.1028
196      2241  0.8269    0.9369 -0.0607
208      2244  0.7950    0.8701 -0.0488
220      2266  0.8220    0.8767 -0.0535

RMSLE medio, 7 orígenes: modelo 0.8239 | zona-mes 0.8982
RMSLE medio, últimos 3 (196/208/220): modelo 0.8147


**Calibración del nivel.** El sesgo (predicción menos real, en log) no tiene signo fijo: es positivo en los años 148 a 184 y negativo en los tres últimos. Es un efecto de nivel común a todas las zonas en cada año, que no se puede anticipar. Como no hay forma de saber qué toca en el test, se aplica una corrección pequeña: la mitad del sesgo medio de los 7 años más la mitad del de los últimos 3. Se limita a un rango estrecho para que no pese más de lo que vale (el techo de ganancia de una calibración perfecta es de unas 0.002).

In [10]:
sesgo_7 = validacion['sesgo'].mean()
sesgo_3 = validacion.loc[[196, 208, 220], 'sesgo'].mean()
DELTA_CALIDAS = float(np.clip(-0.5 * (sesgo_7 + sesgo_3), -0.02, 0.03))
print(f'sesgo medio 7 orígenes: {sesgo_7:+.4f} | últimos 3: {sesgo_3:+.4f} | ajuste aplicado: {DELTA_CALIDAS:+.4f}')

sesgo medio 7 orígenes: +0.0148 | últimos 3: -0.0543 | ajuste aplicado: +0.0198


## 6. Zonas con historia: modelo final

Se entrena con todos los orígenes (hasta `t = 220`, cuyo horizonte termina en `t = 232`) y se promedian cinco semillas, porque `extra_trees` introduce aleatoriedad. Las features del test se calculan con origen `t = 232`.

In [11]:
SEMILLAS = (42, 123, 456, 789, 2024)

filas_test = test[['id', 'zone_id', 'company_id', 't', 'month']]
feat_test = asof_features(train, filas_test, int(train['t'].max()))
feat_test = feat_test.merge(static_df, on='zone_id', how='left')
assert len(feat_test) == len(test) and feat_test['id'].is_unique

pred_calidas_log = entrenar_predecir(ds, feat_test, SEMILLAS)
pred_calidas = pd.Series(pred_calidas_log, index=feat_test['id'].values)
print('predicciones listas:', len(pred_calidas))

predicciones listas: 2715


## 7. Zonas sin historia

Las 42 zonas frías aparecen por primera vez en `t = 233`. Sin historia propia, lo que se puede usar es la empresa a la que pertenecen, la estacionalidad de esa empresa, la vecindad espacial y la infraestructura de la zona.

**Cómo se entrena sin zonas frías en `train`.** Se simulan casos fríos de dos maneras:

- *Nacidas:* zonas que sí aparecen a mitad de la historia; se toman sus primeros 12 meses y se usa como historia todo lo anterior.
- *Leave-zone-out (LZO):* en cada origen, una zona con historia se trata como si fuera nueva: sus 12 meses siguientes se predicen con una historia de la que se la ha quitado.

El modelo no predice el target directamente sino el **residuo** respecto a la media de la empresa en los últimos 24 meses (`c24`), con features relativas a ese nivel. Eso lo hace independiente del nivel de cada época.

In [12]:
def tabla_estatica_frio(substations, consumption):
    """Estáticas reducidas que usa el modelo frío (infraestructura + clientes medios)."""
    sa = (substations.assign(tot=substations.demand_sp + substations.demand_ap)
          .groupby('zone_id').agg(sub_count=('capacity', 'count'), cap_sum=('capacity', 'sum'),
                                  cap_mean=('capacity', 'mean'), dem=('tot', 'sum'),
                                  own=('ownership', lambda s: (s == 'A').mean()),
                                  x=('x', 'mean'), y=('y', 'mean'),
                                  xs=('x', 'std'), ys=('y', 'std')).reset_index())
    sa['stress'] = sa['dem'] / (sa['cap_sum'] + 1e-5)
    tension = (pd.crosstab(substations['zone_id'], substations['voltage_class'], normalize='index')
               .add_prefix('v_').reset_index())
    clientes = (consumption.groupby(['zone_id', 't'])['customers'].sum()
                .groupby('zone_id').mean().rename('cust').reset_index())
    return sa.merge(tension, on='zone_id').merge(clientes, on='zone_id', how='left')


def features_caso(filas, hist, corte, st, zinfo):
    """Features de las filas de UNA zona usando solo `hist` (historia previa a `corte`)."""
    z = filas['zone_id'].iloc[0]
    out = filas.copy()

    # nivel de la empresa en ventanas de 12, 24 y 60 meses
    g24 = hist.loc[hist.t > corte - 24, 'lt'].mean()
    for k in (12, 24, 60):
        medias = hist[hist.t > corte - k].groupby('company_id')['lt'].mean()
        out[f'c{k}'] = out['company_id'].map(medias)
    out['gl24'] = g24
    out['c24'] = out['c24'].fillna(g24)
    out['c12'] = out['c12'].fillna(out['c24'])
    out['c60'] = out['c60'].fillna(out['c24'])

    # estacionalidad de la empresa (10 años) como desvío del nivel medio
    clave = pd.MultiIndex.from_arrays([out['company_id'].values, out['month'].values],
                                      names=['company_id', 'month'])
    h120 = hist[hist.t > corte - 120]
    media_emp = h120.groupby('company_id')['lt'].mean()
    media_emp_mes = h120.groupby(['company_id', 'month'])['lt'].mean()
    est = media_emp_mes - media_emp.reindex(media_emp_mes.index.get_level_values('company_id')).values
    out['seas120'] = np.nan_to_num(est.reindex(clave).values, nan=0.0)

    # prior espacial: media de las 5 zonas más cercanas con al menos 12 meses recientes
    h60 = hist[hist.t > corte - 60]
    zz = h60.groupby('zone_id')['lt'].agg(['mean', 'size']).reset_index()
    zz = zz[zz['size'] >= 12].merge(st[['zone_id', 'x', 'y']], on='zone_id')
    knn = g24
    if z in zinfo.index and np.isfinite(zinfo.loc[z, 'x']) and len(zz) >= 5:
        d = np.hypot(zz['x'].values - zinfo.loc[z, 'x'], zz['y'].values - zinfo.loc[z, 'y'])
        cerca = np.argsort(d)[:5]
        w = 1.0 / (d[cerca] + 1.0)
        knn = float((w * zz['mean'].values[cerca]).sum() / w.sum())
    out['knn'] = knn

    # versiones relativas al nivel de la empresa
    out['dev12'] = out['c12'] - out['c24']
    out['dev60'] = out['c60'] - out['c24']
    out['cg'] = out['c24'] - out['gl24']
    out['knn_rel'] = out['knn'] - out['c24']
    out['sin_m'] = np.sin(2 * np.pi * out['month'] / 12.0)
    out['cos_m'] = np.cos(2 * np.pi * out['month'] / 12.0)
    return out.merge(st, on='zone_id', how='left')


def dataset_frio(train, st):
    """Casos fríos simulados: zonas nacidas a mitad de historia + leave-zone-out por origen."""
    tr = train[['zone_id', 'company_id', 't', 'month', 'target']].copy()
    tr['lt'] = np.log1p(tr['target'].clip(upper=60))
    zinfo = st.set_index('zone_id')
    primera = tr.groupby('zone_id')['t'].min()
    partes = []

    for z in primera.index[primera >= 40]:
        if z not in zinfo.index:
            continue
        t0 = int(primera[z])
        filas = tr[(tr.zone_id == z) & (tr.t >= t0) & (tr.t < t0 + 12)]
        hist = tr[(tr.t < t0) & (tr.t >= t0 - 120)]
        caso = features_caso(filas, hist, t0 - 1, st, zinfo)
        caso['pop'], caso['origin'] = 'nacidas', -1
        partes.append(caso)

    for o in ORIGENES_FRIO:
        hist_o = tr[(tr.t <= o) & (tr.t > o - 120)]
        futuro = tr[(tr.t > o) & (tr.t <= o + 12)]
        for z, filas in futuro.groupby('zone_id'):
            if z not in zinfo.index or primera[z] > o or len(filas) < 9:
                continue
            hist = hist_o[hist_o.zone_id != z]            # la zona no puede verse a sí misma
            caso = features_caso(filas, hist, o, st, zinfo)
            caso['pop'], caso['origin'] = f'LZO{o}', o
            partes.append(caso)
    return pd.concat(partes, ignore_index=True)


ORIGENES_FRIO = list(range(88, 221, 12))                  # 88, 100, ..., 220
st = tabla_estatica_frio(substations, consumption)
C = dataset_frio(train, st)

STAT_FRIO = ['sub_count', 'cap_sum', 'cap_mean', 'dem', 'own', 'x', 'y', 'xs', 'ys', 'stress', 'cust'] + \
            [c for c in C.columns if c.startswith('v_')]
F_FRIO = ['dev12', 'dev60', 'cg', 'seas120', 'knn_rel', 'sin_m', 'cos_m'] + STAT_FRIO
print(f'casos simulados: {len(C)} filas | {C.zone_id.nunique()} zonas únicas | {len(F_FRIO)} features')
print(C.groupby('pop')['zone_id'].nunique().rename('zonas por población').to_string())

casos simulados: 24345 filas | 188 zonas únicas | 24 features
pop
LZO100     131
LZO112     156
LZO124     163
LZO136     162
LZO148     169
LZO160     169
LZO172     169
LZO184     171
LZO196     178
LZO208     180
LZO220     182
LZO88      133
nacidas     77


### Validación del modelo frío

Se usan los tres últimos orígenes LZO (196, 208, 220) como validación. En cada uno se entrena solo con casos de `t <= origen` y **sin las zonas que se están validando** (validación cruzada agrupada por zona, 8 pliegues). El modelo final es el promedio de dos modelos de residuo, un HistGradientBoosting con pesos temporales y un Ridge sobre variables transformadas a normales, con el residuo contraído al 75 % (`S_CONTRACCION`).

In [13]:
S_CONTRACCION = 0.75
LAMBDA_FRIO = 0.01
PRM_HGB = dict(max_iter=120, learning_rate=0.04, max_leaf_nodes=6, min_samples_leaf=40,
               l2_regularization=5.0, random_state=0)


def residuo_hgb(trn, val, origen):
    peso = np.exp(-LAMBDA_FRIO * (origen - trn['t'].values))
    modelo = HistGradientBoostingRegressor(**PRM_HGB)
    modelo.fit(trn[F_FRIO], trn['lt'] - trn['c24'], sample_weight=peso)
    return modelo.predict(val[F_FRIO])


def residuo_ridge(trn, val, origen):
    mediana = trn[F_FRIO].median()
    modelo = make_pipeline(QuantileTransformer(n_quantiles=200, output_distribution='normal', random_state=0),
                           Ridge(alpha=100))
    modelo.fit(trn[F_FRIO].fillna(mediana), trn['lt'] - trn['c24'])
    return modelo.predict(val[F_FRIO].fillna(mediana))


def residuo_ensamble(trn, val, origen):
    return 0.5 * residuo_hgb(trn, val, origen) + 0.5 * residuo_ridge(trn, val, origen)


POPS_VAL = ['LZO196', 'LZO208', 'LZO220']
oof = np.full(len(C), np.nan)
for pop in POPS_VAL:
    o = int(pop.replace('LZO', ''))
    idx = np.where(C['pop'].values == pop)[0]
    cv = C.iloc[idx]
    for _, fuera in GroupKFold(8).split(cv, groups=cv['zone_id']):
        zonas_val = set(cv.iloc[fuera]['zone_id'])
        trn = C[(C['t'] <= o) & (~C['zone_id'].isin(zonas_val))]
        oof[idx[fuera]] = residuo_ensamble(trn, cv.iloc[fuera], o)
    print('listo', pop, flush=True)

listo LZO196


listo LZO208


listo LZO220


In [14]:
y_frio = np.log1p(C['target'].values)
pred_frio_val = C['c24'].values + S_CONTRACCION * oof


def rmsle_pop(pop, pred_log):
    m = C['pop'].values == pop
    return float(np.sqrt(np.mean((en_log(pred_log[m]) - y_frio[m]) ** 2)))


sesgo_frio = {p: float(np.mean(pred_frio_val[C['pop'].values == p] - y_frio[C['pop'].values == p]))
              for p in POPS_VAL}
filas = []
for p in POPS_VAL:
    delta_otros = -float(np.mean([sesgo_frio[q] for q in POPS_VAL if q != p]))    # sin mirar este origen
    filas.append({'origen': p,
                  'empresa 24m': rmsle_pop(p, C['c24'].values),
                  'modelo': rmsle_pop(p, pred_frio_val),
                  'modelo + calibración': rmsle_pop(p, pred_frio_val + delta_otros),
                  'sesgo': sesgo_frio[p]})
validacion_frio = pd.DataFrame(filas).set_index('origen')
print(validacion_frio.round(4).to_string())
print('\npromedio:'); print(validacion_frio.mean().round(4).to_string())

DELTA_FRIAS = -float(np.mean(list(sesgo_frio.values())))
print(f'\najuste de calibración para las frías: {DELTA_FRIAS:+.4f}')

        empresa 24m  modelo  modelo + calibración   sesgo
origen                                                   
LZO196       0.9198  0.8929                0.8908 -0.0613
LZO208       0.9012  0.8653                0.8619 -0.0797
LZO220       0.9194  0.8880                0.8863 -0.0566

promedio:
empresa 24m             0.9135
modelo                  0.8821
modelo + calibración    0.8797
sesgo                  -0.0659

ajuste de calibración para las frías: +0.0659


El sesgo del modelo frío sí tiene el mismo signo en los tres orígenes (el modelo tiende a quedarse corto), así que aquí se aplica el ajuste completo en lugar de uno parcial.

### Predicción de las zonas frías del test

Se entrena con todos los casos simulados y se predicen las 500 filas frías con features calculadas con historia hasta `t = 232`. Una zona que ni siquiera aparece en `substations.csv` se queda solo con la media de su empresa (sin corrección).

In [15]:
corte = int(train['t'].max())
tr_all = train[['zone_id', 'company_id', 't', 'month', 'target']].copy()
tr_all['lt'] = np.log1p(tr_all['target'].clip(upper=60))
hist_test = tr_all[tr_all.t > corte - 120]
zinfo = st.set_index('zone_id')

test_frio = test[test.zone_id.isin(zonas_frias)][['id', 'zone_id', 'company_id', 't', 'month']]
assert len(test_frio) == 500

R = pd.concat([features_caso(f, hist_test, corte, st, zinfo) for _, f in test_frio.groupby('zone_id')],
              ignore_index=True)

# entrenamiento con todos los casos simulados
peso = np.exp(-LAMBDA_FRIO * (corte - C['t'].values))
res = C['lt'] - C['c24']
hgb = HistGradientBoostingRegressor(**PRM_HGB).fit(C[F_FRIO], res, sample_weight=peso)
mediana = C[F_FRIO].median()
ridge = make_pipeline(QuantileTransformer(n_quantiles=200, output_distribution='normal', random_state=0),
                      Ridge(alpha=100)).fit(C[F_FRIO].fillna(mediana), res)

X_frio = R.reindex(columns=F_FRIO)
residuo = 0.5 * hgb.predict(X_frio) + 0.5 * ridge.predict(X_frio.fillna(mediana))

sin_estaticas = (~R['zone_id'].isin(st['zone_id'])).values | R['x'].isna().values
residuo = np.where(sin_estaticas, 0.0, residuo)
print(f'filas sin estáticas (solo media de empresa): {sin_estaticas.sum()} | zonas: {R.loc[sin_estaticas, "zone_id"].nunique()}')

pred_frias_log = R['c24'].values + S_CONTRACCION * residuo + DELTA_FRIAS
pred_frias = pd.Series(np.clip(np.expm1(pred_frias_log), 0.15, 10.0), index=R['id'].values)
print(pred_frias.describe().round(3).to_string())

filas sin estáticas (solo media de empresa): 12 | zonas: 1
count    500.000
mean       2.373
std        0.705
min        1.164
25%        1.825
50%        2.245
75%        2.884
max        4.592


## 8. Armado de la submission

Las zonas con historia salen del modelo LightGBM (con el ajuste de nivel de la sección 5) y las zonas frías del modelo de residuo. Todo se cruza por `id`, nunca por posición.

In [16]:
submission = sample_sub[['id']].copy()
es_fria = submission['id'].isin(set(test_frio['id']))

pred_calidas_final = np.expm1(pred_calidas + DELTA_CALIDAS)
submission['target'] = np.where(es_fria, submission['id'].map(pred_frias),
                                submission['id'].map(pred_calidas_final))

assert len(submission) == 2715 and submission['id'].is_unique
assert set(submission['id']) == set(test['id'])
assert submission['target'].notna().all() and (submission['target'] >= 0).all()
assert es_fria.sum() == 500

submission.to_csv(OUT_DIR / 'submission.csv', index=False)
print(f'guardado outputs/submission.csv | cálidas: {(~es_fria).sum()} | frías: {es_fria.sum()}')
print(submission['target'].describe().round(3).to_string())

guardado outputs/submission.csv | cálidas: 2215 | frías: 500
count    2715.000
mean        2.505
std         1.646
min         0.561
25%         1.325
50%         2.145
75%         3.120
max        13.670


## 9. Lo que se probó y no funcionó

Este modelo es el resultado de descartar bastante. Lo más útil que quedó, por si sirve a alguien más:

**Validación**
- Entrenar con todo `train` y calcular las features con la fila incluida (o mirando al futuro) hace que train y test no se parezcan. Pasar a un dataset multi-origen no mejoró por sí mismo el puntaje, pero permitió validar de forma honesta.
- Un solo origen de validación es demasiado ruidoso. Con siete orígenes se ve que las diferencias menores a ~0.002 no son distinguibles.
- El leaderboard público (30 % de las filas) tampoco sirve para afinar. Se usó solo para detectar errores de alineación.

**Zonas con historia**
- Más rondas empeoran: el error pasa de 0.8268 (100 rondas) a 0.8338 (600). `extra_trees` con regularización fuerte bajó el RMSLE medio de 0.8297 a 0.8239 y gana en los siete orígenes.
- Modelar el residuo respecto a una base de nivel (media de 12 o 24 meses, zona × mes) no ayudó. Las variantes quedan dentro de ±0.002 y sus errores correlacionan >0.99, así que mezclarlas tampoco suma.
- Predecir por componentes (`target_a`, `target_b`, `target_c`) y mezclar con el modelo directo da +0.0003; no justifica cuatro modelos en lugar de uno.
- Quitar las variables estáticas empeora 0.0017. Se dejan.
- Un modelo *hurdle* para los ceros rinde poco: solo el 14.5 % de `target_a` son ceros en los últimos 3 años; los ceros estructurales están en `target_b` y `target_c`.

**Calibración de nivel**
- El sesgo cambia de signo entre años (el nivel medio del target oscila entre 1.07 y 1.27 en escala log). Calibrar con los últimos años es una apuesta. El techo de ganancia es de ~0.002 en cualquier dirección.

**Zonas sin historia**
- La media de empresa por mes es peor que la media de empresa de 24 meses, y el prior espacial (KNN) por sí solo no ayuda. Lo que funcionó fue el modelo de residuo con features relativas.
- Simular zonas nuevas con *leave-zone-out* es la población de validación correcta: reproduce el nivel de error de las frías reales.